# Load datasets into MongoDB

Load JSONL files, preserve or merge splits, and inspect the resulting documents.
Start Docker and run the cells in order with the BONDI environment.
The notebook creates `notebook-datasets` on port 27126. Change these values if they are already in use.
The final cell removes this deployment and the temporary source files.
If execution stops early, run the cleanup cell manually.

See the [dataset guide](../docs/datasets.md) for file formats, external connections and Couchbase conversion rules.

In [ ]:
import json
from pathlib import Path
from tempfile import TemporaryDirectory

from bondi.data import LoadDataError, load_data
from bondi.deployment import deploy_mongodb
from bondi.mongodb import MongoCollectionRef, MongoExecutor, MongoFindQuery

workspace = TemporaryDirectory(prefix="bondi-dataset-demo-")
root = Path(workspace.name)
print(f"Temporary dataset directory: {root}")
deployment = deploy_mongodb(name="notebook-datasets", port=27126)

## Create a split dataset

These files represent one collection with train and test splits.
The explicit namespace keeps generated IDs stable when the files move.
Extended JSON dates become BSON dates in MongoDB.

In [ ]:
train = [
    {"city": "Sydney", "visited": {"$date": "2026-01-01T00:00:00Z"}},
    {"city": "Melbourne", "visited": {"$date": "2026-01-02T00:00:00Z"}},
]
test = [{"city": "Brisbane", "visited": {"$date": "2026-01-03T00:00:00Z"}}]
for split_name, documents in {"train": train, "test": test}.items():
    (root / f"{split_name}.jsonl").write_text(
        "".join(json.dumps(document) + "\n" for document in documents)
    )
manifest = {
    "version": 1,
    "namespace": "dataset-notebook-v1",
    "collections": {"visits": {"splits": {"train": "train.jsonl", "test": "test.jsonl"}}},
}
manifest_path = root / "dataset.json"
manifest_path.write_text(json.dumps(manifest))

## Preserve splits and report progress

Each split gets its own collection. The callback receives a snapshot after each acknowledged batch.
The default write mode refuses nonempty collections.

In [ ]:
def progress(result):
    print("Acknowledged:", result.destinations)


result = load_data(
    manifest=manifest_path,
    target=deployment,
    database="dataset_demo",
    json_mode="extended",
    batch_size=2,
    on_progress=progress,
)
print("Conversions:", result.conversions)
assert result.destinations == {
    "dataset_demo/visits_train": 2,
    "dataset_demo/visits_test": 1,
}

## Inspect the loaded documents

The dates are native datetimes. Generated IDs remain in the documents.
The executor reads all returned rows.

In [ ]:
query = MongoFindQuery(MongoCollectionRef("dataset_demo", "visits_train"), {})
with MongoExecutor.from_uri(deployment.uri) as executor:
    before = executor.run(query, collect_results=True).documents
for document in before:
    print(document)

## Repeat a load explicitly

A second create load fails because the collection contains data.
Use replace to clear the selected split and load it again. The test split remains unchanged.

In [ ]:
try:
    load_data(manifest=manifest_path, target=deployment, database="dataset_demo")
except LoadDataError as error:
    print("Expected refusal:", error)

load_data(
    manifest=manifest_path,
    target=deployment,
    database="dataset_demo",
    json_mode="extended",
    split="train",
    mode="replace",
)
with MongoExecutor.from_uri(deployment.uri) as executor:
    after = executor.run(query, collect_results=True).documents
assert {d["_id"] for d in before} == {d["_id"] for d in after}
print("Generated IDs are unchanged.")

## Merge the splits

Merge loads all three documents into `visits`. It does not add a split field or change generated IDs.
The result still reports source counts for each split.

In [ ]:
merged = load_data(
    manifest=manifest_path,
    target=deployment,
    database="dataset_demo",
    json_mode="extended",
    splits="merge",
)
assert merged.destinations == {"dataset_demo/visits": 3}
print(merged.source_counts)

## Load the packaged dataset

Registered datasets provide their own namespace and source configuration.
Use `bondi datasets list` to discover other registered datasets.
Hub datasets require `bondi[data]` and network access. This example stays local.

In [ ]:
demo = load_data("demo", target=deployment, database="dataset_demo")
print(demo.destinations)
assert demo.destinations == {"dataset_demo/business": 8}

## Remove the demo resources

This deletes the demo deployment, its data volume and its saved configuration.
Calling `remove()` deletes the data volume and saved configuration.
It also removes the temporary source files. Run this cell before restarting the demo from an empty database.

In [ ]:
deployment.remove()
workspace.cleanup()